### Phase 2 prediction

based on logits/final-layer embeddings?

In [ ]:
import re
import os
import math
import json
import torch
import random
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from scipy import sparse
import matplotlib.pyplot as plt
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, utils

Dataset preparation


For logits

In [ ]:
FEATURE_DIR = "./training_data/logits/features/llama3_70b"    # feature directory
METADATA_DIR = "./training_data/logits/metadata/llama3_70b"  # metadata directory

VOCAB_SIZE = 128256  
EMBEDDING_DIM = 1024
HIDDEN_DIM = 512
DROPOUT_RATE = 0.2
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
EPOCHS = 20
TOP_K = 1000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def decode_params(encoded):
    encoded = int(encoded)
    return {
        'temperature': ((encoded >> 24) & 0xFF) / 255 * 0.8 + 0.1,
        'top_k': (encoded >> 17) & 0x7F,
        'repetition_penalty': ((encoded >> 9) & 0xFF) / 255 * 0.3 + 1.3,
        'max_new_tokens': ((encoded >> 6) & 0x7) * 100
    }


class LogitsDataset(Dataset):
    def __init__(self, feature_dir):
        self.samples = []
        for fname in tqdm(os.listdir(feature_dir), desc="Building index"):
            if not fname.endswith('.npz'):
                continue
            path = os.path.join(feature_dir, fname)
            with np.load(path, allow_pickle=True) as data:
                n_samples = len(data['features'])
                for i in range(n_samples):
                    self.samples.append((path, i))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, sample_idx = self.samples[idx]
        with np.load(path, allow_pickle=True) as data:
            feature = data['features'][sample_idx]
            label = data['labels'][sample_idx]
        
        params = decode_params(feature['system_params'].item())
        
        # normalize features
        features = {
            'temperature': np.float32((params['temperature'] - 0.1) / 0.8),
            'top_k': np.float32(params['top_k'] / 100.0),
            'repetition_penalty': np.float32((params['repetition_penalty'] - 1.3) / 0.3),
            'max_len': np.float32((params['max_new_tokens'] - 300) / 200.0),
            'seq_pos': np.float32(feature['seq_pos'].item() / 4096.0),
            'topk_values': torch.tensor(feature['topk_values'], dtype=torch.float32),
            'topk_indices': torch.tensor(feature['topk_indices'], dtype=torch.long),
            'remaining': torch.tensor(label['remaining_tokens'].item(), dtype=torch.float32),
            'over_max': torch.tensor(float(label['over_max_seq_len']), dtype=torch.float32)
        }
        return features

class LogitsRegressor(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout_rate):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.param_encoder = nn.Sequential(
            nn.Linear(4, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, 128)
        )
        
        self.fusion_net = nn.Sequential(
            nn.Linear(embedding_dim + 128 + 1, hidden_dim),  # +1 for seq_pos
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.GELU(),
            nn.Dropout(dropout_rate)
        )
        
        self.reg_head = nn.Sequential(
            nn.Linear(hidden_dim//2, hidden_dim//4),
            nn.GELU(),
            nn.Linear(hidden_dim//4, 1)
        )
        
        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim//2, 1),
            nn.Sigmoid()
        )
    def forward(self, batch):
        # Token feature
        emb = self.embedding(batch['topk_indices'])  # [B, K, E]
        weighted = emb * batch['topk_values'].unsqueeze(-1)
        token_feat = torch.mean(weighted, dim=1)     # [B, E]
        
        # param feature
        params = torch.stack([
            batch['temperature'],
            batch['top_k'],
            batch['repetition_penalty'],
            batch['max_len']
        ], dim=1)
        param_feat = self.param_encoder(params)      # [B, 128]
        
        # Combine features
        combined = torch.cat([
            token_feat,
            param_feat,
            batch['seq_pos'].unsqueeze(1)
        ], dim=1)  # [B, E+128+1]
        
        hidden = self.fusion_net(combined)  # [B, H/2]
        
        return self.reg_head(hidden).squeeze(), self.cls_head(hidden).squeeze()

def train_epoch(model, dataloader, reg_crit, cls_crit, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader, desc="Training"):
        # data to device
        device_batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v 
            for k,v in batch.items()
        }
        reg_labels = device_batch.pop('remaining').to(device)
        cls_labels = device_batch.pop('over_max').to(device)
        
        # forward pass
        optimizer.zero_grad()
        reg_out, cls_out = model(device_batch)
        
        # loss calculation
        reg_loss = reg_crit(reg_out, reg_labels)
        cls_loss = cls_crit(cls_out, cls_labels)
        total_loss = 0.7*reg_loss + 0.3*cls_loss
        
        # backward pass
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    
    return total_loss.item()

def main():
    dataset = LogitsDataset(FEATURE_DIR)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, 
                            num_workers=4, pin_memory=True, persistent_workers=True)
    
    model = LogitsRegressor(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, DROPOUT_RATE)
    model = model.to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, 
                                weight_decay=1e-5, eps=1e-6)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
    
    reg_criterion = nn.HuberLoss(delta=2.0)
    cls_criterion = nn.BCEWithLogitsLoss()
    
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, dataloader, reg_criterion, 
                                cls_criterion, optimizer, DEVICE)
        scheduler.step()
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
    
    torch.save(model.state_dict(), "multi_task_regressor.pth")

if __name__ == "__main__":
    main()

For last layer embeddings

In [ ]:
HIDDEN_SIZE = 4096  # Llama-3隐藏层维度
DROPOUT_RATE = 0.3
BATCH_SIZE = 256    # 可增大batch_size
LEARNING_RATE = 1e-3
EPOCHS = 50         # 需要更多epoch学习embedding特征
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EmbeddingDataset(Dataset):
    def __init__(self, feature_dir):
        self.samples = []
        # 并行加载索引
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=8) as executor:
            futures = []
            for fname in os.listdir(feature_dir):
                if not fname.endswith('.npz'):
                    continue
                path = os.path.join(feature_dir, fname)
                futures.append(executor.submit(self._load_file, path))
            
            for future in tqdm(futures, desc="Loading files"):
                self.samples.extend(future.result())
                
    def _load_file(self, path):
        samples = []
        with np.load(path, allow_pickle=True) as data:
            features = data['features']
            labels = data['labels']
            for feat, lab in zip(features, labels):
                # 解码参数
                params = decode_params(feat['system_params'].item())
                
                # 归一化
                norm_features = {
                    'temperature': (params['temperature'] - 0.1) / 0.8,
                    'top_k': params['top_k'] / 100.0,
                    'repetition_penalty': (params['repetition_penalty'] - 1.3) / 0.3,
                    'max_len': (params['max_new_tokens'] - 300) / 200.0,
                    'seq_pos': feat['seq_pos'].item() / 4096.0,
                    'embedding': feat['embedding'].astype(np.float32),  # float16->float32
                    'remaining': lab['rest_len'],
                    'over_max': float(lab['over_max_len'])
                }
                samples.append(norm_features)
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            'temperature': torch.tensor(sample['temperature'], dtype=torch.float32),
            'top_k': torch.tensor(sample['top_k'], dtype=torch.float32),
            'repetition_penalty': torch.tensor(sample['repetition_penalty'], dtype=torch.float32),
            'max_len': torch.tensor(sample['max_len'], dtype=torch.float32),
            'seq_pos': torch.tensor(sample['seq_pos'], dtype=torch.float32),
            'embedding': torch.tensor(sample['embedding'], dtype=torch.float32),
            'remaining': torch.tensor(sample['remaining'], dtype=torch.float32),
            'over_max': torch.tensor(sample['over_max'], dtype=torch.float32)
        }
        
class EnhancedMLP(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # 特征交互层
        self.interaction = nn.Sequential(
            nn.Linear(hidden_size + 5, 2048),  # 5个参数特征
            nn.BatchNorm1d(2048),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(2048, 1024),
            nn.LayerNorm(1024),
            nn.GELU()
        )
        
        # 多任务输出头
        self.reg_head = nn.Sequential(
            nn.Linear(1024, 512),
            nn.SiLU(),
            nn.Linear(512, 1)
        )
        self.cls_head = nn.Sequential(
            nn.Linear(1024, 256),
            nn.SiLU(),
            nn.Linear(256, 1)
        )
        
        # 残差连接
        self.residual = nn.Linear(hidden_size + 5, 1024)
    
    def forward(self, x):
        # 拼接所有特征
        main_feature = torch.cat([
            x['embedding'],
            x['temperature'].unsqueeze(1),
            x['top_k'].unsqueeze(1),
            x['repetition_penalty'].unsqueeze(1),
            x['max_len'].unsqueeze(1),
            x['seq_pos'].unsqueeze(1)
        ], dim=1)
        
        # 特征交互
        interacted = self.interaction(main_feature)
        
        # 残差连接
        residual = self.residual(main_feature)
        fused = interacted + residual
        
        # 多任务输出
        reg_out = self.reg_head(fused).squeeze()
        cls_out = self.cls_head(fused).squeeze()
        return reg_out, cls_out

def train_step(model, batch, reg_criterion, cls_criterion):
    # 移动数据到设备
    device_batch = {k: v.to(DEVICE) for k, v in batch.items()}
    reg_labels = device_batch.pop('remaining')
    cls_labels = device_batch.pop('over_max')
    
    # 前向传播
    reg_pred, cls_pred = model(device_batch)
    
    # 损失计算
    reg_loss = reg_criterion(reg_pred, reg_labels)
    cls_loss = cls_criterion(cls_pred, cls_labels)
    return 0.6*reg_loss + 0.4*cls_loss

def main():
    # 数据加载
    dataset = EmbeddingDataset(FEATURE_DIR)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=8, pin_memory=True, persistent_workers=True)
    
    # 模型初始化
    model = EnhancedMLP(HIDDEN_SIZE).to(DEVICE)
    
    # 优化配置
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, 
                                weight_decay=1e-4, betas=(0.9, 0.999))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=2e-3, total_steps=EPOCHS*len(dataloader),
        pct_start=0.3
    )
    
    # 损失函数
    reg_criterion = nn.HuberLoss(delta=5.0)  # 允许较大误差范围
    cls_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]).to(DEVICE))  # 处理类别不平衡
    
    # 训练循环
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            optimizer.zero_grad()
            loss = train_step(model, batch, reg_criterion, cls_criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

if __name__ == "__main__":
    main()

For attention_weight things:

- Dataset structure:

    ```json
    {
        "system_params": {
            "temperature": float,
            "top_k": int,
            "repetition_penalty": float,
            "max_seq_len": int
        },
        "model_arch": {
            "num_layers": int,
            "num_heads": int
        },
        "samples": [
            {
                "layer": int,
                "head": int,
                "attention_matrix": np.array(seq_len, seq_len),
                "seq_pos": int,
            },
        ],
        "label": {
            "remaining_tokens": int,
            "over_max_seq_len": bool
        }
    }
    ```
- Task: predict the remaining tokens (and over_max_seq_len) for each sample in the dataset.
  - Regression task
  - Models can be used: MLP, Transformer, RNN, LSTM, GRU, etc.

Here we use **MLP** and train it with the dataset. (maybe incorrect)

In [2]:
from torch.utils.data import Dataset

def commpress_attention_matrix(attention_matrix):
    """
    half precision
    sparse matrix
    Compress the attention matrix by removing the diagonal and upper triangular part
    Not yet: SVG compression
    """
    seq_len = attention_matrix.shape[0]
    mask = np.tri(seq_len, dtype=bool)
    attention_matrix = attention_matrix * mask
    
    attention_matrix = attention_matrix.astype(np.float16)
    sparse_matrix = sparse.coo_matrix(attention_matrix)
    return sparse_matrix

def encode_params(params):
    """
    Encode system parameters into a numpy array / 32-bit integer
    """
    assert 0.1 <= params['temperature'] <= 0.9
    assert 1 <= params['top_k'] <= 255
    assert 1.0 <= params['repetition_penalty'] <= 1.6
    assert 100 <= params['max_seq_len'] <= 25500
    
    # encode
    temp_enc = int(np.interp(params['temperature'], [0.1, 0.9], [0, 255]))
    topk_enc = params['top_k']
    rep_enc = int(np.interp(params['repetition_penalty'], [1.0, 1.6], [0, 255]))
    maxlen_enc = params['max_seq_len'] // 100
    
    return np.uint32(
        (temp_enc << 24) | 
        (topk_enc << 16) | 
        (rep_enc << 8) | 
        maxlen_enc
    )

def decode_params(encoded):
    """
    Decode system parameters from a numpy array / 32-bit integer
    """
    return {
        'temperature': ((encoded >> 24) & 0xFF) / 255 * 0.8 + 0.1,
        'top_k': (encoded >> 16) & 0xFF,
        'repetition_penalty': ((encoded >> 8) & 0xFF) / 255 * 0.6 + 1.0,
        'max_seq_len': (encoded & 0xFF) * 100
    }

class CompressedDataset(Dataset):
    def __init__(self, feature_dir):
        self.samples = []
        for fname in os.listdir(feature_dir):
            data = np.load(os.path.join(feature_dir, fname), allow_pickle=True)
            for feature, label in zip(data['features'], data['labels']):
                self.samples.append((feature, label))
                
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        feature, label = self.samples[idx]
        
        # 解码参数
        params = decode_params(feature['encoded_params'])
        
        # 重建注意力矩阵
        attn_matrix = sparse.coo_matrix(
            (feature['attn_data'], 
            (feature['attn_row'], feature['attn_col'])),
            shape=feature['attn_shape']
        ).toarray().astype(np.float32)
        
        # 构建训练样本
        return {
            'attn_matrix': torch.tensor(attn_matrix),
            'layer': torch.tensor(feature['layer']),
            'head': torch.tensor(feature['head']),
            'seq_pos': torch.tensor(feature['seq_pos'] / params['max_seq_len']),
            'temperature': torch.tensor(params['temperature']),
            'top_k': torch.tensor(params['top_k']),
            'rep_penalty': torch.tensor(params['repetition_penalty']),
            'label': torch.tensor(label['remaining_tokens'])
        }